# Testing HuggingFace Connection with a Simple Call

In [ ]:
# HuggingFace Test

from huggingface_hub import InferenceClient

HUGGINFACEHUB_API_TOKEN = os.environ["HUGGINGFACEHUB_API_TOKEN_2"]

client = InferenceClient(
    provider="together",
    api_key=HUGGINFACEHUB_API_TOKEN,
)

messages = [
	{
		"role": "user",
		"content": "What is the capital of France?"
	}
]

completion = client.chat.completions.create(
    model="meta-llama/Llama-3.2-3B-Instruct", 
	messages=messages, 
	max_tokens=500,
)

print(completion.choices[0].message)

ChatCompletionOutputMessage(role='assistant', content='The capital of France is Paris.', tool_calls=[])


In [14]:
print(completion.choices[0].message)

ChatCompletionOutputMessage(role='assistant', content='```json\n{\n    "city": "Istanbul",\n    "user_taste": "You seem to prefer cities with a rich history and cultural heritage, often opting for lesser-known destinations over popular tourist spots. You tend to enjoy exploring unique neighborhoods and discovering hidden gems, such as quaint cafes, local markets, and historic landmarks, which suggests that you value authenticity and character in the places you visit.",\n    "recommendations": [\n        { \n            "name": "Çukurcuma",\n            "description": "A charming neighborhood filled with antique shops, vintage cafes, and historic buildings",\n            "address": "Çukurcuma, Beyoğlu, Istanbul",\n            "category": "local neighborhood"\n        },\n        { \n            "name": "Karaköy Güllüoğlu",\n            "description": "A historic baklava shop with stunning views of the Golden Horn",\n            "address": "Karaköy, Beyoğlu, Istanbul",\n            "cate

## HugginFace Endpoints + Langchain

In [5]:
from langchain_huggingface import HuggingFaceEndpoint
from prompt_config import  SYSTEM, HUMAN
from langchain.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, PromptTemplate
from langchain.chains import LLMChain

import os

repo_id = "mistralai/Mistral-7B-Instruct-v0.2"
HUGGINFACEHUB_API_TOKEN = os.environ["HUGGINGFACEHUB_API_TOKEN_2"]

system_template_prompt = SystemMessagePromptTemplate.from_template(SYSTEM)
human_template_prompt = HumanMessagePromptTemplate.from_template(HUMAN, input_variables=["preferences", "city"])
prompt_template = ChatPromptTemplate.from_messages([
        system_template_prompt,
        human_template_prompt
        ]
    )

llm = HuggingFaceEndpoint(
    repo_id=repo_id,
    max_new_tokens=4800,
    temperature=0.5,
    huggingfacehub_api_token=HUGGINFACEHUB_API_TOKEN,
)

# Chain them together
chain = LLMChain(llm=llm, prompt=prompt_template)
print(prompt_template)

preferences = """They preferred Uttrecht over Amsterdam. They preferred s-Hertogenbosch over Eindhoven. They preferred Barcelona over Lisbon.
They preffered Belgrade over Zagreb. They preffered Barcelona over all other cities."""
city = "Istanbul"
input_data = {"preferences": preferences, "city": city}

# Run the chain
response = chain.run(input_data)
print(response)


input_variables=['city', 'preferences'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful local travel guide that always responds in valid JSON format.\n            Help a customer to find the next bars, cafes, museums, coffee shops, shops, landmarks, remarkable or historical districts, local neigbourhoods, Breweries/Distilleries/Wineries, Local Markets, outdoor areas like parks and gardens, local events etc in their next travel based on their preferences. \n            Avoid popular spots and tourist traps; prefer hidden gems like a local guide would do and suggest at least 10 spots.\n            Your response MUST be in valid JSON format. The JSON should be a single object with the following structure:\n\n            EXAMPLE_FORMAT = ```json\n            {{city: name of the city,\n\n            user_taste: explain user’s taste and what they prefer in two

/home/sam/Documents/Codez/ss/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'post' (from 'huggingface_hub.inference._client') is deprecated and will be removed from version '0.31.0'. Making direct POST requests to the inference server is not supported anymore. Please use task methods instead (e.g. `InferenceClient.chat_completion`). If your use case is not supported, please open an issue in https://github.com/huggingface/huggingface_hub.
  warnings.warn(warning_message, FutureWarning)



    Here's an example of how you might fill in the JSON for a user who prefers historic districts and local markets:

    ```json
    {
        "city": "Istanbul",
        "user_taste": "You enjoy exploring historic districts and vibrant local markets. Each new place is an opportunity to immerse yourself in the local culture and history.",
        "recommendations": [
            {
                "name": "Balat",
                "description": "A historic neighborhood located on the European side of Istanbul. Balat is famous for its colorful houses, beautiful churches, and synagogues.",
                "address": "Balat, Istanbul",
                "category": "historical district"
            },
            {
                "name": "Baltalimanı Carsi",
                "description": "A bustling local market located in the Eminönü district. Baltalimanı Carsi offers a wide variety of fresh produce, spices, textiles, and souvenirs.",
                "address": "Eminönü, Istanbul",
    

## HuggingFace Endpoints + Langchain + Pydantic

In [6]:
from langchain.output_parsers import PydanticOutputParser
from models import Response, Recommendations

parser = PydanticOutputParser(pydantic_object=Response)

In [7]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"$defs": {"Recommendations": {"properties": {"name": {"description": "name of the spot", "title": "Name", "type": "string"}, "description": {"description": "short description of the spot", "title": "Description", "type": "string"}, "address": {"description": "open address of the location", "title": "Address", "type": "string"}, "category": {"description": "category of the spot", "title": "Category", "type": "string"}}, "required": ["name", "description", "address", "category"], "title": "Recommendations", "type": "object"}}, "properti

In [9]:
my_response = parser.parse(response)

In [12]:
my_response.__dir__()

['city',
 'user_taste',
 'recommendations',
 '__module__',
 '__annotations__',
 'model_config',
 '__class_vars__',
 '__private_attributes__',
 '__weakref__',
 '__doc__',
 '__abstractmethods__',
 '_abc_impl',
 '__pydantic_custom_init__',
 '__pydantic_post_init__',
 '__pydantic_decorators__',
 '__pydantic_generic_metadata__',
 '__pydantic_complete__',
 '__pydantic_parent_namespace__',
 '__pydantic_fields__',
 '__pydantic_core_schema__',
 '__pydantic_validator__',
 '__pydantic_serializer__',
 '__signature__',
 '__pydantic_computed_fields__',
 '__pydantic_root_model__',
 '__slots__',
 '__init__',
 'model_fields',
 'model_computed_fields',
 'model_extra',
 'model_fields_set',
 'model_construct',
 'model_copy',
 'model_dump',
 'model_dump_json',
 'model_json_schema',
 'model_parametrized_name',
 'model_post_init',
 'model_rebuild',
 'model_validate',
 'model_validate_json',
 'model_validate_strings',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__pydantic_init_subclass

In [13]:
my_response.recommendations

[Recommendations(name='Balat', description='A historic neighborhood located on the European side of Istanbul. Balat is famous for its colorful houses, beautiful churches, and synagogues.', address='Balat, Istanbul', category='historical district'),
 Recommendations(name='Baltalimanı Carsi', description='A bustling local market located in the Eminönü district. Baltalimanı Carsi offers a wide variety of fresh produce, spices, textiles, and souvenirs.', address='Eminönü, Istanbul', category='local market'),
 Recommendations(name='Karaköy Lokantası', description='A traditional Turkish restaurant located in the Karaköy district. Karaköy Lokantası is known for its delicious meze dishes and friendly service.', address='Karaköy, Istanbul', category='restaurant'),
 Recommendations(name='Rumeli Hisarı', description='A historic fortress located on the European side of the Bosphorus. Rumeli Hisarı offers stunning views of the Bosphorus and the sea.', address='Rumeli Hisarı, Istanbul', category='hi

In [14]:
my_response.dict()

/tmp/ipykernel_7587/700698875.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  my_response.dict()


{'city': 'Istanbul',
 'user_taste': 'You enjoy exploring historic districts and vibrant local markets. Each new place is an opportunity to immerse yourself in the local culture and history.',
 'recommendations': [{'name': 'Balat',
   'description': 'A historic neighborhood located on the European side of Istanbul. Balat is famous for its colorful houses, beautiful churches, and synagogues.',
   'address': 'Balat, Istanbul',
   'category': 'historical district'},
  {'name': 'Baltalimanı Carsi',
   'description': 'A bustling local market located in the Eminönü district. Baltalimanı Carsi offers a wide variety of fresh produce, spices, textiles, and souvenirs.',
   'address': 'Eminönü, Istanbul',
   'category': 'local market'},
  {'name': 'Karaköy Lokantası',
   'description': 'A traditional Turkish restaurant located in the Karaköy district. Karaköy Lokantası is known for its delicious meze dishes and friendly service.',
   'address': 'Karaköy, Istanbul',
   'category': 'restaurant'},
  